# 第7章：构建聊天应用
## OpenAI API 快速入门

本笔记本改编自[Azure OpenAI 示例代码库](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)，其中包含访问 [Azure OpenAI](notebook-azure-openai.ipynb) 服务的笔记本。

Python OpenAI API 也可与 Azure OpenAI 模型一起使用，只需进行少量修改。了解更多差异信息，请参阅：[如何在 Python 中切换 OpenAI 和 Azure OpenAI 端点](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# 概述  
“大型语言模型是将文本映射到文本的函数。给定一个输入的文本字符串，大型语言模型尝试预测接下来将出现的文本”(1)。这个“快速入门”笔记本将向用户介绍高级的LLM概念、开始使用AML所需的核心软件包、提示设计的基础介绍，以及几个不同使用场景的简短示例。 


## 目录  

[概述](#overview)  
[如何使用 OpenAI 服务](#how-to-use-openai-service)  
[1. 创建你的 OpenAI 服务](#1.-creating-your-openai-service)  
[2. 安装](#2.-installation)    
[3. 凭证](#3.-credentials)  

[用例](#use-cases)    
[1. 总结文本](#1.-summarize-text)  
[2. 分类文本](#2.-classify-text)  
[3. 生成新产品名称](#3.-generate-new-product-names)  
[4. 微调分类器](#4.fine-tune-a-classifier)  

[参考文献](#references)


### 构建你的第一个提示  
这个简短的练习将为向 OpenAI 模型提交用于简单任务“摘要”的提示提供基本介绍。


<strong>步骤</strong>：  
1. 在你的 Python 环境中安装 OpenAI 库  
2. 加载标准辅助库，并为你创建的 OpenAI 服务设置典型的 OpenAI 安全凭证  
3. 为你的任务选择一个模型  
4. 为模型创建一个简单的提示  
5. 向模型 API 提交你的请求！


### 1. 安装 OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. 导入辅助库并实例化凭据


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. 寻找合适的模型  
类似 GPT-4o 和 GPT-4o mini 这样的模型能够理解和生成自然语言，并且在 OpenAI 平台上提供，具有不同的性能和速度水平，适合不同的任务。


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. 提示设计  

“大型语言模型的神奇之处在于，通过在大量文本上训练以最小化预测误差，模型最终学会了对这些预测有用的概念。例如，它们学会了”(1):

* 如何拼写
* 语法如何运作
* 如何改写
* 如何回答问题
* 如何进行对话
* 如何用多种语言写作
* 如何编程
* 等等。

#### 如何控制大型语言模型  
“在所有输入到大型语言模型的内容中，影响最大的是文本提示(1)。

可以通过几种方式提示大型语言模型输出：

指令：告诉模型你想要什么
补全：引导模型完成你想要开始的内容
示范：向模型展示你想要什么，包括：
提示中的几个示例
微调训练数据集中数百或数千个示例”



#### 创建提示的三个基本准则：

<strong>展示并说明</strong>。通过指令、示例或两者结合，明确你想要的内容。如果你想让模型按字母顺序排列一列列表，或按情感对段落进行分类，就向它展示你想要的结果。

<strong>提供高质量数据</strong>。如果你试图构建分类器或让模型遵循某种模式，确保有足够的示例。务必校对你的示例——模型通常足够智能，能够识别基本拼写错误并给出响应，但它也可能认为这是一种刻意，进而影响响应结果。

**检查你的设置。** temperature 和 top_p 设置控制模型生成响应的确定性。如果你要求模型回答只有一个正确答案的问题，最好将这些值设低。如果你想获得更多样化的响应，可以将它们设高。人们对这些设置最常犯的错误是假设它们是“聪明度”或“创造力”的控制。


来源：https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. 提交！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### 重复相同的调用，结果如何比较？ 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## 总结文本  
#### 挑战  
总结文本，通过在文本段落末尾添加“tl;dr:”来实现。注意模型如何在没有额外指令的情况下理解并执行多项任务。您可以尝试使用比 tl;dr 更具描述性的提示词来调整模型行为并定制您获得的摘要(3)。  

最近的研究表明，通过在大量文本语料上进行预训练，随后在特定任务上进行微调，可以在许多自然语言处理任务和基准测试中取得显著进展。虽然通常架构对任务是无差别的，但此方法仍需要成千上万甚至数万个特定任务的微调数据集。相比之下，人类通常只需少量示例或简单指令就能完成新的语言任务——这是当前的自然语言处理系统仍然很难做到的。本文展示了扩大语言模型规模极大地改善了无任务依赖的少样本性能，有时甚至能达到先前微调方法的竞争水平。 



摘要  


# 多种用例的练习  
1. 文本总结  
2. 文本分类  
3. 生成新产品名称


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 文本分类  
#### 挑战  
将项目分类到推理时提供的类别中。在以下示例中，我们在提示中同时提供了类别和要分类的文本(*playground_reference)。 

客户询问：您好，我笔记本键盘上的一个键最近坏了，我需要更换一个：

分类类别：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## 生成新产品名称
#### 挑战
从示例词汇中创建产品名称。这里我们在提示中包含了关于我们将要为其生成名称的产品的信息。我们还提供了类似的示例，以展示我们希望收到的模式。我们还将温度值设置得较高，以增加随机性和更具创新性的响应。

产品描述：家用奶昔机
种子词：快速，健康，紧凑。
产品名称：HomeShaker, Fit Shaker, QuickShake, Shake Maker

产品描述：一双可以适应任何脚尺码的鞋子。
种子词：适应性，合脚，全尺码。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# 参考资料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [微调 GPT 模型以分类文本的最佳实践](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 更多帮助  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# 贡献者
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
